# Predicting Building Energy Loads

Frizza Mae G. Cale, Julea Kriszel M. Guzman, Annika Ruzzette A. Lomugdang

Department of Architecture, Mapúa Malayan Colleges Mindanao

This notebook contains the executable workflow behind the presentation. The recorded findings below come from a completed run of the identical companion Python workflow. Notebook code cells are ready for you to execute. The data are simulated, and the model has no Philippine field validation.

## 1. Install dependencies
Run once in Colab or Jupyter. Skip this cell if the exact versions are already installed. Restart the kernel if the package manager asks you to.

In [10]:
# # @title
# %pip install -q numpy==2.3.5 pandas==2.2.3 scikit-learn==1.8.0 joblib==1.5.3 threadpoolctl==3.6.0 openpyxl==3.1.5 matplotlib==3.10.8

In [11]:
import numpy as np
import scipy
import sklearn
import pandas as pd

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pd.__version__)
print("All imports successful")

NumPy: 2.1.3
SciPy: 1.16.3
Scikit-learn: 1.6.1
Pandas: 2.2.3
All imports successful


## 2. Define the reproducible experiment
This cell defines the training code. It does not train until the next cell. Training uses fold-specific preprocessing, separate models for heating and cooling, and nested validation for unseen shapes.

In [12]:
"""Reproducible UCI Energy Efficiency experiment. Run: python train_models.py.
All reported metrics are measured against simulated responses, not field energy use.
"""
from pathlib import Path
import argparse, hashlib, json, os, platform, sys, time, urllib.request, zipfile
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
import numpy as np
import pandas as pd
import sklearn
import joblib
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, KFold, GroupKFold, LeaveOneGroupOut, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from threadpoolctl import threadpool_limits
ROOT=Path.cwd() / 'energy_study'
ROOT.mkdir(parents=True, exist_ok=True)
FEATURES=['X'+str(i) for i in range(1,9)]
CONT=['X1','X2','X3','X4','X5','X7']; CAT=['X6','X8']
SEED=42
URL='https://archive.ics.uci.edu/static/public/242/energy+efficiency.zip'
NAMES={'X1':'Relative compactness','X2':'Surface area','X3':'Wall area','X4':'Roof area','X5':'Overall height','X6':'Orientation','X7':'Glazing area','X8':'Glazing distribution'}
def dump(obj,path):
 Path(path).write_text(json.dumps(obj,indent=2,default=lambda x:x.item() if hasattr(x,'item') else str(x)))
def load_data(root=ROOT):
 d=root/'data';d.mkdir(exist_ok=True,parents=True);f=d/'ENB2012_data.xlsx'
 if not f.exists():
  urllib.request.urlretrieve(URL,d/'energy_efficiency.zip')
  with zipfile.ZipFile(d/'energy_efficiency.zip') as z:f.write_bytes(z.read('ENB2012_data.xlsx'))
 df=pd.read_excel(f).dropna(how='all')
 assert list(df.columns)==FEATURES+['Y1','Y2'], 'Unexpected columns'
 assert df.shape==(768,10), 'Unexpected dataset size'
 assert not df.isna().any().any(), 'Missing data require an explicit policy'
 assert not df[FEATURES].duplicated().any(), 'Duplicate predictors need grouped splitting'
 groups=pd.factorize(pd.MultiIndex.from_frame(df[FEATURES[:5]]),sort=True)[0]
 assert len(np.unique(groups))==12
 return df,groups,f

def make_model(name):
 prep=ColumnTransformer([('continuous',StandardScaler() if name=='MLR' else 'passthrough',CONT),('categorical',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),CAT)])
 model={'MLR':LinearRegression(),'DT':DecisionTreeRegressor(random_state=SEED),'RF':RandomForestRegressor(n_estimators=100,random_state=SEED,n_jobs=1)}[name]
 return Pipeline([('preprocess',prep),('model',model)])
def grid(name):
 if name=='MLR':return {}
 g={'model__max_depth':[3,5,10,None],'model__min_samples_leaf':[1,2,5]}
 if name=='RF':g['model__max_features']=[0.5,1.0]
 return g

def metrics(y,p):
 return {'MAE':float(mean_absolute_error(y,p)),'RMSE':float(np.sqrt(mean_squared_error(y,p))),'R2':float(r2_score(y,p))}

def tune(name,X,y,cv,jobs,groups=None):
 search=GridSearchCV(make_model(name),grid(name),cv=cv,scoring={'mae':'neg_mean_absolute_error','rmse':'neg_root_mean_squared_error','r2':'r2'},refit='mae',n_jobs=jobs,error_score='raise',return_train_score=True)
 start=time.perf_counter();search.fit(X,y,groups=groups);elapsed=time.perf_counter()-start
 return search,elapsed

def timing(model,X,y,test):
 fits=[]
 for _ in range(5):
  t=time.perf_counter();clone(model).fit(X,y);fits.append(1000*(time.perf_counter()-t))
 row=test.iloc[[0]]
 for _ in range(100):model.predict(row)
 calls=[]
 for _ in range(1000):
  t=time.perf_counter();model.predict(row);calls.append(1000*(time.perf_counter()-t))
 batch=[]
 for _ in range(30):
  t=time.perf_counter();model.predict(test);batch.append(1000*(time.perf_counter()-t))
 return {'fit_median_ms':float(np.median(fits)),'fit_iqr_ms':float(np.subtract(*np.percentile(fits,[75,25]))),'single_median_ms':float(np.median(calls)),'single_iqr_ms':float(np.subtract(*np.percentile(calls,[75,25]))),'batch_median_ms':float(np.median(batch))}

def generate_plots(root=ROOT):
 """Create diagnostic charts from saved predictions without refitting models."""
 import matplotlib
 matplotlib.use('Agg')
 import matplotlib.pyplot as plt
 out=root/'results';figdir=out/'figures';figdir.mkdir(exist_ok=True)
 pp=pd.read_csv(out/'test_predictions.csv')
 for target,label in [('Y1','Heating'),('Y2','Cooling')]:
  fig,axes=plt.subplots(2,3,figsize=(13,7))
  for j,name in enumerate(['MLR','DT','RF']):
   g=pp[(pp.target==target)&(pp.model==name)]
   lo=min(g.actual.min(),g.predicted.min());hi=max(g.actual.max(),g.predicted.max())
   axes[0,j].scatter(g.actual,g.predicted,s=15,alpha=.65,color='#167D8D')
   axes[0,j].plot([lo,hi],[lo,hi],color='#BD6739',linestyle='--')
   axes[0,j].set(title=name,xlabel='Simulated load',ylabel='Predicted load')
   axes[1,j].scatter(g.predicted,g.predicted-g.actual,s=15,alpha=.65,color='#167D8D')
   axes[1,j].axhline(0,color='#BD6739',linestyle='--')
   axes[1,j].set(xlabel='Predicted load',ylabel='Prediction minus simulated load')
  fig.suptitle(label+' load: held-out predictions and residuals (dataset units)')
  fig.tight_layout();fig.savefig(figdir/(target+'_diagnostics.png'),dpi=160);plt.close(fig)
 g=pd.read_csv(out/'shape_fold_results.csv')
 fig,axes=plt.subplots(1,2,figsize=(13,4))
 for ax,target,label in zip(axes,['Y1','Y2'],['Heating','Cooling']):
  for name,color in [('MLR','#163347'),('DT','#BD6739'),('RF','#167D8D')]:
   h=g[(g.target==target)&(g.model==name)].sort_values('shape_group')
   ax.plot(h.shape_group,h.MAE,marker='o',label=name,color=color)
  ax.set(title=label,xlabel='Excluded shape group',ylabel='MAE (dataset units)');ax.legend()
 fig.tight_layout();fig.savefig(figdir/'unseen_shape_errors.png',dpi=160);plt.close(fig)

def main(jobs=2,root=ROOT):
 start=time.perf_counter();out=root/'results';out.mkdir(exist_ok=True);(root/'models').mkdir(exist_ok=True)
 df,groups,file=load_data(root);X=df[FEATURES]
 train,test=train_test_split(np.arange(len(df)),test_size=.2,random_state=SEED)
 assert not set(train)&set(test)
 splits=pd.DataFrame({'row_id':np.arange(768),'shape_group':groups,'partition':['development' if i in set(train) else 'test' for i in range(768)]});splits.to_csv(out/'splits.csv',index=False)
 df[FEATURES].corr().to_csv(out/'feature_correlations.csv')
 info={'source_url':URL,'source_doi':'10.24432/C51307','data_sha256':hashlib.sha256(file.read_bytes()).hexdigest(),'rows':len(df),'features':FEATURES,'shapes':12,'development_n':len(train),'test_n':len(test),'seed':SEED,'python':sys.version,'sklearn':sklearn.__version__,'numpy':np.__version__,'pandas':pd.__version__,'joblib':joblib.__version__,'platform':platform.platform(),'cpu':platform.processor(),'logical_cpus':os.cpu_count(),'grid_jobs':jobs,'estimator_jobs':1,'units':'Original dataset load units; repository does not specify physical units.','missing_values':int(df.isna().sum().sum()),'duplicate_predictors':int(df[FEATURES].duplicated().sum())}
 dump(info,out/'run_metadata.json')
 rows=[];predictions=[];selected={};models={};importance=[]
 cv=KFold(n_splits=5,shuffle=True,random_state=SEED)
 for target in ['Y1','Y2']:
  y=df[target]
  dummy=DummyRegressor().fit(X.iloc[train],y.iloc[train]);dp=dummy.predict(X.iloc[test]);rows.append({'target':target,'model':'Mean baseline',**metrics(y.iloc[test],dp)})
  for name in ['MLR','DT','RF']:
   search,elapsed=tune(name,X.iloc[train],y.iloc[train],cv,jobs)
   model=search.best_estimator_;models[target,name]=model;idx=search.best_index_;res=search.cv_results_
   pred=model.predict(X.iloc[test]);ms=timing(model,X.iloc[train],y.iloc[train],X.iloc[test])
   row={'target':target,'model':name,**metrics(y.iloc[test],pred),'cv_MAE':-res['mean_test_mae'][idx],'cv_MAE_sd':res['std_test_mae'][idx],'cv_RMSE':-res['mean_test_rmse'][idx],'cv_R2':res['mean_test_r2'][idx],'train_MAE':mean_absolute_error(y.iloc[train],model.predict(X.iloc[train])),'search_seconds':elapsed,**ms,'parameters':search.best_params_}
   rows.append(row);joblib.dump(model,root/'models'/f'{target}_{name}_development.joblib')
   pd.DataFrame(res).to_csv(out/f'cv_{target}_{name}.csv',index=False)
   predictions.extend({'row_id':int(r),'target':target,'model':name,'actual':float(y.iloc[r]),'predicted':float(p)} for r,p in zip(test,pred))
   print('PRIMARY',target,name,json.dumps({k:row[k] for k in ['MAE','RMSE','R2','cv_MAE']}),flush=True)
  candidates=[r for r in rows if r['target']==target and r['model']!='Mean baseline'];best=min(r['cv_MAE'] for r in candidates)
  eligible=[r for r in candidates if r['cv_MAE']<=1.01*best]
  winner=min(eligible,key=lambda r:(r['cv_RMSE'],r['single_median_ms'],['MLR','DT','RF'].index(r['model'])))['model'];selected[target]=winner
  pi=permutation_importance(models[target,winner],X.iloc[test],y.iloc[test],scoring='neg_mean_absolute_error',n_repeats=30,random_state=SEED,n_jobs=jobs)
  importance.extend({'target':target,'model':winner,'feature':f,'label':NAMES[f],'MAE_increase':float(m),'sd':float(s)} for f,m,s in zip(FEATURES,pi.importances_mean,pi.importances_std))
 dump(selected,out/'selected_models.json');dump(rows,out/'primary_results.json');pd.DataFrame(rows).to_csv(out/'primary_results.csv',index=False);pd.DataFrame(predictions).to_csv(out/'test_predictions.csv',index=False);pd.DataFrame(importance).to_csv(out/'permutation_importance.csv',index=False)
 # Nested validation by complete building shape. No held-out-shape tuning.
 group_rows=[];group_predictions=[];logo=LeaveOneGroupOut()
 for target in ['Y1','Y2']:
  y=df[target]
  for fold,(tr,te) in enumerate(logo.split(X,y,groups)):
   assert not set(groups[tr])&set(groups[te])
   for name in ['MLR','DT','RF']:
    search,elapsed=tune(name,X.iloc[tr],y.iloc[tr],GroupKFold(3),jobs,groups[tr]);p=search.predict(X.iloc[te])
    group_rows.append({'target':target,'model':name,'shape_group':int(groups[te][0]),'n':len(te),**metrics(y.iloc[te],p),'parameters':search.best_params_})
    group_predictions.extend({'row_id':int(r),'shape_group':int(groups[r]),'target':target,'model':name,'actual':float(y.iloc[r]),'predicted':float(v)} for r,v in zip(te,p))
   print('GROUP',target,fold+1,'/12 complete',flush=True)
   pd.DataFrame(group_rows).to_csv(out/'shape_fold_results.csv',index=False)
   pd.DataFrame(group_predictions).to_csv(out/'shape_predictions.csv',index=False)
 group_df=pd.DataFrame(group_predictions);summary=[]
 for (target,name),g in group_df.groupby(['target','model']):
  fs=pd.DataFrame(group_rows);fs=fs[(fs.target==target)&(fs.model==name)]
  summary.append({'target':target,'model':name,**metrics(g.actual,g.predicted),'shape_MAE_sd':float(fs.MAE.std(ddof=1)),'shape_MAE_min':float(fs.MAE.min()),'shape_MAE_max':float(fs.MAE.max())})
 dump(summary,out/'shape_results.json');pd.DataFrame(summary).to_csv(out/'shape_results.csv',index=False)
 # Predeclared demonstration: first lexicographic matched geometry/orientation/
 # glazing-distribution group with >=3 glazing values, selected without outcomes.
 match=['X1','X2','X3','X4','X5','X6','X8']
 candidates=[g for _,g in df.groupby(match,sort=True) if g.X7.nunique()>=3]
 demo=candidates[0].sort_values('X7').copy();demo.insert(0,'row_id',demo.index)
 for target in ['Y1','Y2']:
  pp=group_df[(group_df.target==target)&(group_df.model==selected[target])].set_index('row_id').predicted
  demo[target+'_predicted']=demo.row_id.map(pp)
 demo.to_csv(out/'design_demonstration.csv',index=False)
 generate_plots(root)
 info['total_seconds']=time.perf_counter()-start;info['completed']=True;dump(info,out/'run_metadata.json')
 print('COMPLETE',selected,'seconds',info['total_seconds'],flush=True)
 return {'primary':rows,'shape':summary,'selected':selected}



## 3. Run the experiment or inspect your saved run
The full experiment can take several minutes. If saved results already exist, this cell loads them. Set `RERUN = True` to repeat everything. A first run downloads the UCI dataset. Keep the 154 test records out of training and tuning.

In [13]:
RERUN = True
if RERUN or not (ROOT/'results/shape_results.json').exists():
    with threadpool_limits(limits=1):
        study = main(jobs=2, root=ROOT)
else:
    print('Using existing results:', ROOT/'results')
primary = pd.read_csv(ROOT/'results/primary_results.csv')
shape_results = pd.read_csv(ROOT/'results/shape_results.csv')
display(primary[['target','model','MAE','RMSE','R2']])
display(shape_results[['target','model','MAE','RMSE','R2']])

PRIMARY Y1 MLR {"MAE": 2.059036233338561, "RMSE": 2.8722770936820705, "R2": 0.9208501870029356, "cv_MAE": 2.0313465909816957}
PRIMARY Y1 DT {"MAE": 0.44608194186765626, "RMSE": 0.663748644734146, "R2": 0.9957732698838883, "cv_MAE": 0.3730465184296653}
PRIMARY Y1 RF {"MAE": 0.35574111373628814, "RMSE": 0.5204149326785124, "R2": 0.9974016555501228, "cv_MAE": 0.3457950270072702}
PRIMARY Y2 MLR {"MAE": 2.180511908625353, "RMSE": 3.1197549334643533, "R2": 0.89495833056301, "cv_MAE": 2.311369065609322}
PRIMARY Y2 DT {"MAE": 1.264618742368742, "RMSE": 1.949091606218951, "R2": 0.9589998772001237, "cv_MAE": 1.1512573275390374}
PRIMARY Y2 RF {"MAE": 1.2020668831168837, "RMSE": 1.9028200144737715, "R2": 0.960923462318751, "cv_MAE": 1.0437114414234305}
GROUP Y1 1 /12 complete
GROUP Y1 2 /12 complete
GROUP Y1 3 /12 complete
GROUP Y1 4 /12 complete
GROUP Y1 5 /12 complete
GROUP Y1 6 /12 complete
GROUP Y1 7 /12 complete
GROUP Y1 8 /12 complete
GROUP Y1 9 /12 complete
GROUP Y1 10 /12 complete
GROUP Y1

,target,model,MAE,RMSE,R2
0,Y1,Mean baseline,9.272103,10.237590,-0.005525
1,Y1,MLR,2.059036,2.872277,0.920850
2,Y1,DT,0.446082,0.663749,0.995773
3,Y1,RF,0.355741,0.520415,0.997402
4,Y2,Mean baseline,8.741853,9.666248,-0.008409
5,Y2,MLR,2.180512,3.119755,0.894958
6,Y2,DT,1.264619,1.949092,0.959000
7,Y2,RF,1.202067,1.902820,0.960923


,target,model,MAE,RMSE,R2
0,Y1,DT,5.789582,8.848181,0.230030
1,Y1,MLR,3.473115,4.722868,0.780630
2,Y1,RF,4.154618,5.942949,0.652648
3,Y2,DT,6.836753,9.277205,0.047780
4,Y2,MLR,3.926464,5.304207,0.688725
5,Y2,RF,4.842097,6.115050,0.586283


## 4. Interpret the measured results
Recorded companion-run findings: RF held-out MAE was 0.356 for heating and 1.202 for cooling. In nested unseen-shape evaluation, MLR had the lowest pooled MAE, 3.473 and 3.926. Use your rerun tables as the source of truth. Small numerical changes can occur across environments. Error units remain the original dataset load units. R² is not percentage accuracy.

In [14]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for ax, target, title in zip(axes, ['Y1','Y2'], ['Heating load','Cooling load']):
    p=primary[(primary.target==target)&(primary.model!='Mean baseline')].set_index('model')
    g=shape_results[shape_results.target==target].set_index('model')
    table=pd.DataFrame({'Random holdout':p.MAE,'Unseen shape':g.MAE}).loc[['MLR','DT','RF']]
    table.plot.bar(ax=ax,rot=0,color=['#16697a','#c87041'])
    ax.set_title(title);ax.set_ylabel('MAE (dataset load units)');ax.set_xlabel('')
plt.tight_layout();plt.show()

## 5. Make a bounded prediction
This example uses a recorded held-out configuration. The saved models were fitted to the development split. Do not replace these values with an arbitrary real building and treat the output as a validated estimate.

In [15]:
df, groups, source_file = load_data(ROOT)
splits = pd.read_csv(ROOT/'results/splits.csv')
row_id = int(splits.loc[splits.partition=='test','row_id'].iloc[0])
example = df.loc[[row_id], FEATURES]
selected = json.loads((ROOT/'results/selected_models.json').read_text())
print('Input record:', row_id)
display(example)
for target, method in selected.items():
    fitted = joblib.load(ROOT/'models'/f'{target}_{method}_development.joblib')
    value = float(fitted.predict(example)[0])
    print(target, 'predicted:', round(value,3), 'simulated:', df.loc[row_id,target])

Input record: 2


,X1,X2,X3,X4,X5,X6,X7,X8
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0


Y1 predicted: 15.964 simulated: 15.55
Y2 predicted: 22.398 simulated: 21.33


## 6. Inspect model settings and architectural interpretation
Permutation importance measures the increase in error after shuffling an input. Correlated geometric features complicate interpretation. Low orientation importance here does not establish that orientation is unimportant for real buildings.

In [16]:
display(primary[['target','model','cv_MAE','cv_MAE_sd','parameters']].dropna(subset=['cv_MAE']))
display(pd.read_csv(ROOT/'results/permutation_importance.csv'))
display(pd.read_csv(ROOT/'results/design_demonstration.csv'))

,target,model,cv_MAE,cv_MAE_sd,parameters
1,Y1,MLR,2.031347,0.087224,{}
2,Y1,DT,0.373047,0.020050,"{'model__max_depth': 10, 'model__min_samples_l..."
3,Y1,RF,0.345795,0.018845,"{'model__max_depth': 10, 'model__max_features'..."
5,Y2,MLR,2.311369,0.146376,{}
6,Y2,DT,1.151257,0.083558,"{'model__max_depth': 10, 'model__min_samples_l..."
7,Y2,RF,1.043711,0.119130,"{'model__max_depth': None, 'model__max_feature..."


,target,model,feature,label,MAE_increase,sd
0,Y1,RF,X1,Relative compactness,4.715567,0.324477
1,Y1,RF,X2,Surface area,2.128985,0.120225
2,Y1,RF,X3,Wall area,1.135192,0.119294
3,Y1,RF,X4,Roof area,1.392037,0.126871
4,Y1,RF,X5,Overall height,1.015512,0.097197
5,Y1,RF,X6,Orientation,0.000052,0.009272
6,Y1,RF,X7,Glazing area,2.696671,0.187234
7,Y1,RF,X8,Glazing distribution,0.067101,0.016924
8,Y2,RF,X1,Relative compactness,2.575494,0.170796
9,Y2,RF,X2,Surface area,0.991250,0.073650


,row_id,X1,X2,X3,X4,X5,X6,X7,X8,Y1,Y2,Y1_predicted,Y2_predicted
0,92,0.62,808.5,367.5,220.5,3.5,2,0.10,1,12.96,14.34,15.429508,19.395950
1,332,0.62,808.5,367.5,220.5,3.5,2,0.25,1,15.09,15.77,17.497020,20.952777
2,572,0.62,808.5,367.5,220.5,3.5,2,0.40,1,15.09,17.20,19.414350,22.583767


## 7. Export your rerun
This cell packages the data, fitted models and results generated in your session. In Colab, download the ZIP from the Files sidebar.

In [17]:
import shutil
archive = shutil.make_archive(str(ROOT.parent/'Building_Energy_Study_Rerun'), 'zip', ROOT.parent, ROOT.name)
print(archive)

/content/Building_Energy_Study_Rerun.zip


## Sources
Dataset: https://doi.org/10.24432/C51307 (CC BY 4.0).

Associated paper: https://doi.org/10.1016/j.enbuild.2012.03.003.

Validation: https://scikit-learn.org/stable/modules/cross_validation.html.

Preprocessing: https://scikit-learn.org/stable/common_pitfalls.html.

Model interpretation: https://scikit-learn.org/stable/modules/permutation_importance.html.